# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/-Explaining-Search-Performance-Gaps-Using-Ranking-Signals/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window



In [1]:
import duckdb
import google.colab.userdata

con = duckdb.connect()

hf_token = google.colab.userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"


con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      9841378 │
└──────────────┘



In [4]:
import duckdb
import google.colab.userdata

con = duckdb.connect()

hf_token = google.colab.userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")


grain_check_query = """
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5;
"""

df_grain = con.execute(grain_check_query).df()
print("duplication  rows :", len(df_grain))
display(df_grain)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplication  rows : 0


,client_hash_id,content_hash_id,report_date,row_count


### Unit of Analysis Definition:

*One row in my dataset slice represents the daily search and engagement performance metrics for a single content item (content_hash_id) belonging to a specific client (client_hash_id) on a specific day (report_date).*

In [5]:
stats_query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet';
"""

df_stats = con.execute(stats_query).df()
display(df_stats)

,total_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
import duckdb
import google.colab.userdata

con = duckdb.connect()
hf_token = google.colab.userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# Query to extract 5 safe features for the Ranking lane while verifying GA4 data availability
features_query = """
SELECT
    f.client_hash_id,
    f.content_hash_id,

    -- Aggregate over the month (or your chosen window)
    AVG(feat_ctr) AS avg_ctr,
    SUM(feat_impressions) AS total_impressions,
    AVG(feat_avg_position) AS avg_position,
    AVG(feat_engagement_rate) AS avg_engagement_rate,
    MAX(feat_word_count) AS word_count  -- static, doesn't change daily

FROM (
    -- Subquery with daily rows + GA4 filter
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        (f.gsc_clicks * 1.0 / NULLIF(f.gsc_impressions, 0)) AS feat_ctr,
        f.gsc_impressions AS feat_impressions,
        f.gsc_avg_position AS feat_avg_position,
        f.ga4_engagement_rate AS feat_engagement_rate,
        COALESCE(c.word_count, 0) AS feat_word_count
    FROM '...fact_content_daily_performance/month=2026-03/*.parquet' f
    LEFT JOIN '...dim_content/*.parquet' c
        ON f.content_hash_id = c.content_hash_id
    WHERE f.ga4_data_available IS TRUE
) daily_rows
GROUP BY f.client_hash_id, f.content_hash_id
LIMIT 10000;
"""

df_features = con.execute(features_query).df()
print("Surviving rows after GA4 availability filter:", len(df_features))
display(df_features.head())

IOException: IO Error: No files found that match the pattern "...fact_content_daily_performance/month=2026-03/*.parquet"

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.